# Stage 7 — Ablations

Three ablations, one per run (set `EXPERIMENT` in the configuration cell; decision D18):

- `"learning_curve"`: QLoRA on 250, 500 and 1,000 human labels (nested, stratified subsets of
  the 1,500), same recipe as Stage 5. The 1,500 point is the Stage 5 run.
- `"seeds"`: QLoRA on the same 1,500 human labels with training seeds 43 and 44 (LoRA
  initialization and example order). Seed 42 is the Stage 5 run.
- `"few_shot"`: the base model in fp16 (no training) with 6-shot prompts whose examples are
  (a) human-labeled, from the 1,500, and (b) two new draws of star-labeled examples.

Everything is measured on the **500 selection reviews** of dev. **The test set is not used**: it
was evaluated once in Stage 6 (D12, D17). Nothing here changes the published model.

Runs on **Google Colab** or **Kaggle** (*GPU T4 x2*, *Internet on*, *Save & Run All*).

In [ ]:
import importlib.util
import os
import subprocess
import sys

import torch

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    PLATFORM, WORK_DIR, OUTPUT_DIR = "kaggle", "/tmp", "/kaggle/working"
elif importlib.util.find_spec("google.colab") is not None:
    PLATFORM, WORK_DIR, OUTPUT_DIR = "colab", "/content", "/content"
else:
    raise RuntimeError("Run this notebook on Google Colab or Kaggle")
print("platform:", PLATFORM)

assert torch.cuda.is_available(), "No GPU: choose a T4 GPU in the notebook settings"
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME)

REPO_URL = "https://github.com/PedroRicci12/ptbr-review-lora.git"
REPO_DIR = f"{WORK_DIR}/ptbr-review-lora"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository commit:", REPO_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes"], check=True)

In [ ]:
subprocess.run(
    [sys.executable, "scripts/prepare_data.py"],
    check=True, stdout=subprocess.DEVNULL, env={**os.environ, "PYTHONPATH": "src"},
)
diff = subprocess.run(["git", "diff", "--exit-code", "reports/data_summary.json"], capture_output=True, text=True)
assert diff.returncode == 0, "Prepared data differs from the repository:\n" + diff.stdout
print("data_summary.json identical to the repository")

## Configuration

**Set `EXPERIMENT` before running.** The recipe is the one of Stage 5 (D16), not tuned.

In [ ]:
EXPERIMENT = "learning_curve"   # "learning_curve", "seeds" or "few_shot"
assert EXPERIMENT in ("learning_curve", "seeds", "few_shot")

MODEL_ID = "HuggingFaceTB/SmolLM3-3B"
MODEL_REVISION = "a07cc9a04f16550a088caea529712d1d335b0ac1"

SEED = 42                      # data splits and subsets (same as Stages 4-5)
USE_TITLE = True
DEV_TRAIN_SIZE = 1500
EPOCHS = 3                     # as the human-label run of Stage 5
MAX_TRAIN_TOKENS = 512
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LEARNING_RATE, WARMUP_RATIO = 2e-4, 0.03
MICRO_BATCH, GRAD_ACCUM, MAX_GRAD_NORM = 4, 4, 1.0
CHECKPOINTS = (0.25, 0.5, 0.75, 1.0)
SCORE_BATCH = 8
N_RESAMPLES = 1000

LC_SIZES = (250, 500, 1000)    # learning curve; 1,500 comes from Stage 5
TRAIN_SEEDS = (43, 44)         # seed variance; 42 comes from Stage 5
STAR_DRAW_SEEDS = (1, 2)       # few-shot: new draws of star-labeled examples (42 was Stage 4)

import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ptbr_review_lora.data import LABELS, model_input, split_dev
from ptbr_review_lora.metrics import bootstrap_ci, compute_metrics, macro_f1, neutral_recall_by_group, paired_bootstrap
from ptbr_review_lora.predictions import load_predictions, save_predictions
from ptbr_review_lora.prompts import build_messages, choose_label, sample_few_shot
from ptbr_review_lora.training import build_example, nested_subsets, pad_batch


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


pool = pd.read_parquet("data/processed/b2w_pool.parquet")
dev = pd.read_parquet("data/processed/repro_dev.parquet")
dev_train, select = split_dev(dev, train_size=DEV_TRAIN_SIZE, seed=SEED)
y_select = select["label"].tolist()
SELECT_ROWS = select.index.tolist()
select_texts = model_input(select, USE_TITLE).tolist()
print("dev_train:", len(dev_train), "selection:", len(select))

## Shared pieces: prompts, label scoring, evaluation

Same prompt (no date, thinking off) and same label scoring as Stages 4-5.

In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
LABEL_IDS = [tokenizer(label, add_special_tokens=False)["input_ids"] for label in LABELS]
ANSWER_IDS = {label: ids + [END_OF_TURN_ID] for label, ids in zip(LABELS, LABEL_IDS)}


def render(review, examples=()):
    messages = build_messages(review, examples)
    messages[0]["content"] += "\n/system_override"  # SmolLM3: no date in the prompt
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)


def encode_prompts(texts, examples=()):
    return [tokenizer(render(t, examples), add_special_tokens=False)["input_ids"] for t in texts]


@torch.inference_mode()
def score_batch(decoder, lm_head, prompts):
    seqs = [prompt + label_ids for prompt in prompts for label_ids in LABEL_IDS]
    input_ids = torch.full((len(seqs), max(map(len, seqs))), PAD_ID)
    attention_mask = torch.zeros_like(input_ids)
    rows, positions, targets = [], [], []
    for s, seq in enumerate(seqs):
        input_ids[s, : len(seq)] = torch.tensor(seq)
        attention_mask[s, : len(seq)] = 1
        prompt_len = len(prompts[s // len(LABELS)])
        for k, token in enumerate(LABEL_IDS[s % len(LABELS)]):
            rows.append(s)
            positions.append(prompt_len - 1 + k)
            targets.append(token)
    with torch.autocast("cuda", dtype=torch.float16):
        hidden = decoder(input_ids=input_ids.cuda(), attention_mask=attention_mask.cuda()).last_hidden_state
        rows_t = torch.tensor(rows, device="cuda")
        logits = lm_head(hidden[rows_t, torch.tensor(positions, device="cuda")])
    log_probs = torch.log_softmax(logits.float(), dim=-1)
    token_scores = log_probs[torch.arange(len(rows), device="cuda"), torch.tensor(targets, device="cuda")]
    scores = torch.zeros(len(seqs), device="cuda").index_add_(0, rows_t, token_scores)
    return scores.view(len(prompts), len(LABELS)).cpu().numpy()


def predict(model, decoder, lm_head, prompts):
    was_training = model.training
    model.eval()
    order = np.argsort([-len(p) for p in prompts])
    scores = np.empty((len(prompts), len(LABELS)))
    for start in range(0, len(prompts), SCORE_BATCH):
        idx = order[start : start + SCORE_BATCH]
        scores[idx] = score_batch(decoder, lm_head, [prompts[i] for i in idx])
    model.train(was_training)
    assert np.isfinite(scores).all(), "non-finite scores"
    return [choose_label(dict(zip(LABELS, row))) for row in scores]


def evaluate(preds):
    _, low, high = bootstrap_ci(y_select, preds, n_resamples=N_RESAMPLES, seed=SEED)
    return {
        **compute_metrics(y_select, preds),
        "macro_f1_ci95": [low, high],
        "neutral_recall_by_group": neutral_recall_by_group(select["polarity_group"], preds),
    }


def reference(stem, folder="dev"):
    """Saved predictions of an earlier model, restricted to the selection rows."""
    if folder == "dev_select":
        return load_predictions(Path(f"predictions/dev_select/{stem}.csv"), len(select))
    preds = load_predictions(Path(f"predictions/dev/{stem}.csv"), len(dev))
    return [preds[r] for r in SELECT_ROWS]


REFS = {
    "zero_shot": reference("smollm3_base__zero_shot"),
    "6_shot_stars_seed42": reference("smollm3_base__6_shot"),
    "qlora_human_1500_seed42": reference("smollm3_qlora_human", folder="dev_select"),
}
print({name: round(macro_f1(y_select, p), 4) for name, p in REFS.items()})

## QLoRA training (learning curve and seeds)

Same as Stage 5: 4-bit NF4 base, LoRA r 16 on all linear layers, loss on the answer tokens only,
checkpoints at 25/50/75/100% of training, best one kept. The model is reloaded for every run so
each starts from the untouched base.

In [ ]:
def load_qlora(train_seed):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import BitsAndBytesConfig

    set_seed(train_seed)  # LoRA initialization depends on the torch seed
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, quantization_config=bnb,
                                                 dtype=torch.float16, device_map={"": 0})
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                             target_modules="all-linear", bias="none", task_type="CAUSAL_LM"))
    base = model.get_base_model()
    return model, base.get_decoder(), base.get_output_embeddings()


def train_run(train_df, train_seed):
    """One QLoRA run; returns its history, best step and selection-set predictions of the best checkpoint."""
    from peft import get_peft_model_state_dict, set_peft_model_state_dict
    from transformers import get_linear_schedule_with_warmup

    model, decoder, lm_head = load_qlora(train_seed)
    texts, labels = model_input(train_df, USE_TITLE).tolist(), train_df["label"].tolist()
    examples = [build_example(p, ANSWER_IDS[y]) for p, y in zip(encode_prompts(texts), labels)]
    n_long = sum(len(e["input_ids"]) > MAX_TRAIN_TOKENS for e in examples)
    examples = [e for e in examples if len(e["input_ids"]) <= MAX_TRAIN_TOKENS]

    steps_per_epoch = len(examples) // (MICRO_BATCH * GRAD_ACCUM)
    total_steps = EPOCHS * steps_per_epoch
    eval_at = sorted({max(1, round(f * total_steps)) for f in CHECKPOINTS})
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.0)
    scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(WARMUP_RATIO * total_steps)), total_steps)
    scaler = torch.amp.GradScaler("cuda")

    def answer_loss(batch):
        input_ids = torch.tensor(batch["input_ids"], device="cuda")
        attention_mask = torch.tensor(batch["attention_mask"], device="cuda")
        labels_t = torch.tensor(batch["labels"], device="cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            hidden = decoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            targets = labels_t[:, 1:]
            keep = targets != -100
            logits = lm_head(hidden[:, :-1][keep])
        return torch.nn.functional.cross_entropy(logits.float(), targets[keep])

    select_prompts = encode_prompts(select_texts)
    rng = np.random.default_rng(train_seed)  # example order depends on the training seed
    history, checkpoints, preds_at = [], {}, {}
    step, losses, start = 0, [], time.perf_counter()
    model.train()
    for _ in range(EPOCHS):
        order = rng.permutation(len(examples))
        micro = [order[i : i + MICRO_BATCH] for i in range(0, steps_per_epoch * MICRO_BATCH * GRAD_ACCUM, MICRO_BATCH)]
        for m, idx in enumerate(micro):
            loss = answer_loss(pad_batch([examples[i] for i in idx], PAD_ID)) / GRAD_ACCUM
            scaler.scale(loss).backward()
            losses.append(loss.item() * GRAD_ACCUM)
            if (m + 1) % GRAD_ACCUM:
                continue
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            if step in eval_at:
                preds = predict(model, decoder, lm_head, select_prompts)
                history.append({"step": step, "train_loss": float(np.mean(losses)), "select_macro_f1": macro_f1(y_select, preds),
                                "elapsed_seconds": round(time.perf_counter() - start)})
                losses, preds_at[step] = [], preds
                print("  ", history[-1])
    best = max(history, key=lambda h: h["select_macro_f1"])
    run = {"train_seed": train_seed, "training_examples": len(examples), "dropped_long_examples": n_long,
           "total_steps": total_steps, "history": history, "best_step": best["step"],
           "train_seconds": round(time.perf_counter() - start)}
    best_preds = preds_at[best["step"]]

    del model, decoder, lm_head, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return run, best_preds

## Run the experiment

In [ ]:
runs, run_preds = {}, {}
start_all = time.perf_counter()

if EXPERIMENT == "learning_curve":
    subsets = nested_subsets(dev_train, (*LC_SIZES, DEV_TRAIN_SIZE), seed=SEED)
    for size in LC_SIZES:
        name = f"qlora_human_{size}_seed42"
        print(name, subsets[size]["label"].value_counts().to_dict())
        runs[name], run_preds[name] = train_run(subsets[size].sample(frac=1, random_state=SEED), train_seed=SEED)
        runs[name]["label_counts"] = subsets[size]["label"].value_counts().to_dict()

elif EXPERIMENT == "seeds":
    train_df = dev_train.sample(frac=1, random_state=SEED)  # same data and order source as Stage 5
    for train_seed in TRAIN_SEEDS:
        name = f"qlora_human_1500_seed{train_seed}"
        print(name)
        runs[name], run_preds[name] = train_run(train_df, train_seed=train_seed)

else:  # few_shot: base model in fp16, no training
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16).cuda().eval()
    decoder, lm_head = model.get_decoder(), model.get_output_embeddings()
    example_sets = {"6_shot_human_seed42": sample_few_shot(dev_train, per_class=2, seed=SEED, use_title=USE_TITLE)}
    for draw in STAR_DRAW_SEEDS:
        example_sets[f"6_shot_stars_seed{draw}"] = sample_few_shot(pool, per_class=2, seed=draw, use_title=USE_TITLE)
    for name, ex in example_sets.items():
        t0 = time.perf_counter()
        prompts = encode_prompts(select_texts, list(zip(ex["text"], ex["label"])))
        run_preds[name] = predict(model, decoder, lm_head, prompts)
        # Rows are dev positions (human examples) or pool positions (star examples); no text is stored.
        runs[name] = {"example_rows": ex.index.tolist(), "example_labels": ex["label"].tolist(),
                      "example_source": "repro_dev_train" if "human" in name else "b2w_pool",
                      "seconds": round(time.perf_counter() - t0)}
        print(name, round(macro_f1(y_select, run_preds[name]), 4), runs[name]["seconds"], "s")

TOTAL_SECONDS = round(time.perf_counter() - start_all)
print("total", TOTAL_SECONDS, "s")

## Results on the selection set

Each run is compared with the Stage 5 model (1,500 human labels, seed 42) and with zero-shot, on
the same 500 reviews (paired bootstrap, 95%). These are exploratory: no correction for multiple
comparisons, and no decision depends on them.

In [ ]:
results, comparisons = {}, {}
for name, preds in {**run_preds, **REFS}.items():
    results[name] = evaluate(preds)
for name, preds in run_preds.items():
    for ref in ("qlora_human_1500_seed42", "zero_shot") if EXPERIMENT != "few_shot" else ("zero_shot", "6_shot_stars_seed42"):
        comparisons[f"{name} vs {ref}"] = paired_bootstrap(y_select, REFS[ref], preds, n_resamples=N_RESAMPLES, seed=SEED)

summary = {}
if EXPERIMENT == "seeds":
    f1s = [results["qlora_human_1500_seed42"]["macro_f1"]] + [results[n]["macro_f1"] for n in run_preds]
    summary = {"seeds": [42, *TRAIN_SEEDS], "macro_f1": f1s, "mean": float(np.mean(f1s)), "std": float(np.std(f1s, ddof=1))}
    print("seed variance:", summary)

for name, r in results.items():
    print(f"{name}: macro-F1 {r['macro_f1']:.4f} (95% CI {r['macro_f1_ci95'][0]:.4f}-{r['macro_f1_ci95'][1]:.4f}), "
          f"F1 neutro {r['f1_per_class']['neutro']:.3f}")
for name, c in comparisons.items():
    print(f"{name}: diff {c['diff']:+.4f} (95% CI {c['low']:+.4f} to {c['high']:+.4f})")

In [ ]:
import platform
import zipfile

import sklearn

env = {"platform": PLATFORM, "gpu": GPU_NAME, "python": platform.python_version(), "torch": torch.__version__,
       "transformers": transformers.__version__, "scikit_learn": sklearn.__version__, "pandas": pd.__version__}
if EXPERIMENT != "few_shot":
    import bitsandbytes
    import peft

    env |= {"peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__}

report = {
    "experiment": EXPERIMENT, "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "repo_commit": REPO_COMMIT,
    "environment": env, "split": "repro_dev_select", "n": len(select), "n_resamples": N_RESAMPLES,
    "runs": runs, "results": results, "paired_comparisons": comparisons, "summary": summary,
    "total_seconds": TOTAL_SECONDS,
}
report_path = Path(f"reports/ablation_{EXPERIMENT}_select.json")
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
files_to_pack = [str(report_path)]
for name, preds in run_preds.items():
    path = Path(f"predictions/dev_select/ablation__{name}.csv")
    save_predictions(path, preds)
    files_to_pack.append(str(path))

ZIP_PATH = f"{OUTPUT_DIR}/ablation_{EXPERIMENT}_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_pack:
        zf.write(path)
print("saved", ZIP_PATH)
if PLATFORM == "colab":
    from google.colab import files

    files.download(ZIP_PATH)
else:
    print("Kaggle: download it from the Output tab of this notebook version")